# Kang et al. 2018: multi-sample validation notebook

An **independent** implementation of the multi-sample analysis described in
`docs/kang_validation_spec.md` (2026-11-11). It was written only from that spec, the data and the public
documentation of Scanpy and harmonypy, without looking at the app's code, so the app can be checked against it.

It writes the expected answers to `validation/expected_multisample/` (or to `$ORDITUS_EXPORT_DIR` if set):
`kang_summary.json` and `kang_results.h5ad`.

**Choices made where the spec left room** (also listed in the summary under `choices`):
1. The count matrix is kept in the format it is stored in (CSC sparse, float32); no conversion.
2. The summary records the date but no time and no output path, so two runs on the same day give identical files.
3. Spec section 4 step 3 says "the QC metrics from step 2"; this is read as section 2's metrics (computed once,
   before any filtering).
4. For the Harmony runs, the mixing score per cluster is reported both before (original PCs) and after
   (Harmony coordinates) correction (analyst's choice, 2026-11-11).
5. `obs["condition"]` is categorical with categories in order of first appearance, like `obs["sample"]`.
6. Interferon means are computed in float64 from the float32 log-normalized values.
7. harmonypy gets the PCA coordinates as Scanpy stores them (float32) and the `obs` table with the
   `sample` column; `verbose` is left at its default (it only prints).

In [1]:
import datetime
import hashlib
import json
import math
import os
import platform
from importlib.metadata import version
from pathlib import Path

import anndata as ad
import harmonypy
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
from scipy.stats import median_abs_deviation
from sklearn.metrics import adjusted_rand_score
from sklearn.neighbors import NearestNeighbors

ROOT = Path.cwd().parent if Path.cwd().name == "reference" else Path.cwd()
DATA_FILE = ROOT / "data" / "kang_2018.h5ad"
OUT_DIR = Path(os.environ.get("ORDITUS_EXPORT_DIR", ROOT / "validation" / "expected_multisample"))
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 0
SETTINGS = {
    "samples": {"donor_column": "replicate", "condition_column": "label", "name_separator": " · "},
    "qc_metrics": {"mito_prefix": "MT-", "qc_vars": ["mt"], "percent_top": None, "log1p": False},
    "qc_limits": {"genes_mads": 5, "pct_mt_mads": 3, "mad_scale": 1.0,
                  "min_genes_rule": "max(0, ceil(expm1(median - N*MAD))) on log1p(n_genes_by_counts); keep >=",
                  "max_genes_rule": "floor(expm1(median + N*MAD)) + 1; keep <",
                  "max_pct_mt_rule": "ceil((median + 3*MAD) * 10) / 10; keep <; no limit without mito genes",
                  "mad_zero": "no gene limits for that sample"},
    "filter": {"min_cells_per_gene": 3, "order": ["min_genes per sample", "min_cells", "max_genes (+ max_pct_mt) per sample"]},
    "normalize": {"target_sum": None, "log1p": True},
    "hvg": {"flavor": "seurat", "min_mean": 0.0125, "max_mean": 3, "min_disp": 0.5,
            "variants": {"pooled": {"batch_key": None}, "per_sample": {"batch_key": "sample"}}},
    "scale": {"regress_out": None, "max_value": 10},
    "pca": {"n_comps": 50, "svd_solver": "arpack", "random_state": SEED},
    "harmony": {"function": "harmonypy.run_harmony", "n_pcs": 50, "vars_use": "sample", "theta": 2,
                "max_iter_harmony": 10, "random_state": SEED, "other": "harmonypy defaults"},
    "neighbors": {"n_neighbors": 15, "n_pcs": 50, "use_rep": "X_pca or X_pca_harmony", "random_state": SEED},
    "leiden": {"flavor": "leidenalg", "resolution": 1.0, "n_iterations": -1, "random_state": SEED,
               "directed": True, "run_before_umap": True},
    "umap": {"random_state": SEED, "other": "Scanpy defaults"},
    "markers": {"groupby": "leiden", "method": "wilcoxon", "use_raw": True, "pts": True, "reference": "rest",
                "corr_method": "benjamini-hochberg", "tie_correct": False, "top_n_saved": 25},
    "mixing": {"k": 30, "metric": "euclidean", "search": "exact (sklearn NearestNeighbors, brute)",
               "self_excluded": True, "log": "natural", "batch": "sample", "pcs": 50},
    "checks": {"isg_genes": ["ISG15", "IFI6", "IFIT1", "IFIT3", "MX1"], "min_cells_per_condition": 50},
}
CHOICES = [
    "Count matrix kept as stored (CSC sparse, float32); no format conversion.",
    "Summary records the date only (no time, no output path) so same-day runs are identical.",
    "Spec section 4 step 3 'QC metrics from step 2' read as section 2's metrics (computed once, before filtering).",
    "Harmony runs: mixing per cluster reported both before and after correction (analyst's choice, 2026-11-11).",
    "obs['condition'] is categorical in order of first appearance.",
    "Interferon means computed in float64 from the float32 log-normalized values.",
    "harmonypy receives X_pca as stored by Scanpy (float32) and obs with the 'sample' column; verbose left at default.",
]

PACKAGES = ["scanpy", "anndata", "numpy", "scipy", "pandas", "scikit-learn", "umap-learn",
            "pynndescent", "leidenalg", "igraph", "harmonypy"]
VERSIONS = {"python": platform.python_version(), **{p: version(p) for p in PACKAGES}}
print(VERSIONS)
print("Writing to", OUT_DIR)

{'python': '3.11.17', 'scanpy': '1.11.5', 'anndata': '0.12.19', 'numpy': '2.4.6', 'scipy': '1.17.1', 'pandas': '2.3.3', 'scikit-learn': '1.9.1', 'umap-learn': '0.5.12', 'pynndescent': '0.6.0', 'leidenalg': '0.12.0', 'igraph': '1.0.0', 'harmonypy': '2.1.0'}
Writing to /Users/aditivannela/orditus/pbmc analysis app/validation/expected_multisample


## 1. Data

Load the raw counts, record the file's SHA-256 and check the counts are non-negative whole numbers.
A **sample** is one donor (`replicate`) under one condition (`label`), named `"<replicate> · <label>"`.
The authors' `cell_type` is kept only for the biological checks at the end. The file also carries the authors'
own `X_pca` / `X_umap`; they are removed so nothing downstream can use them.

In [2]:
def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()

INPUT_SHA256 = sha256_of(DATA_FILE)
adata = sc.read_h5ad(DATA_FILE)
print(adata)
print("X:", type(adata.X).__name__, adata.X.dtype)

values = adata.X.data if sp.issparse(adata.X) else np.asarray(adata.X)
assert np.all(values >= 0), "counts must be non-negative"
assert np.all(values == np.floor(values)), "counts must be whole numbers"

names_were_unique = bool(adata.var_names.is_unique)
if not names_were_unique:
    adata.var_names_make_unique()
assert adata.var_names.is_unique and adata.obs_names.is_unique

for key in list(adata.obsm.keys()):
    del adata.obsm[key]

donor = adata.obs["replicate"].astype(str)
label = adata.obs["label"].astype(str)
sample_names = donor + " · " + label
adata.obs["sample"] = pd.Categorical(sample_names, categories=list(pd.unique(sample_names)))
adata.obs["condition"] = pd.Categorical(label, categories=list(pd.unique(label)))

# Every donor has cells under both conditions (spec), so samples = donors x conditions.
pairs = adata.obs.groupby(["replicate", "label"], observed=True).size()
assert len(pairs) == donor.nunique() * label.nunique()
assert adata.obs["sample"].cat.categories.size == len(pairs)
assert set(label.unique()) == {"ctrl", "stim"}

condition_of = dict(zip(sample_names, label))
SAMPLES = [{"name": s, "condition": condition_of[s], "cells_loaded": int((sample_names == s).sum())}
           for s in adata.obs["sample"].cat.categories]
N_CELLS_LOADED, N_GENES_LOADED = adata.shape
pd.DataFrame(SAMPLES)

AnnData object with n_obs × n_vars = 24673 × 15706
    obs: 'nCount_RNA', 'nFeature_RNA', 'tsne1', 'tsne2', 'label', 'cluster', 'cell_type', 'replicate', 'nCount_SCT', 'nFeature_SCT', 'integrated_snn_res.0.4', 'seurat_clusters'
    var: 'name'
    obsm: 'X_pca', 'X_umap'
X: csc_matrix float32


,name,condition,cells_loaded
0,patient_1016 · ctrl,ctrl,1723
1,patient_1256 · ctrl,ctrl,2108
2,patient_1488 · ctrl,ctrl,2031
3,patient_1039 · ctrl,ctrl,461
4,patient_1015 · ctrl,ctrl,2738
5,patient_107 · ctrl,ctrl,517
6,patient_1244 · ctrl,ctrl,1879
7,patient_101 · ctrl,ctrl,858
8,patient_101 · stim,stim,1166
9,patient_1488 · stim,stim,2549


## 2. Quality control metrics

Mitochondrial genes are those whose names start with `MT-` (case-sensitive). The QC metrics are computed
**once**, on all cells and genes, before any filtering, and never recomputed.

In [3]:
adata.var["mt"] = adata.var_names.str.startswith("MT-")
N_MITO = int(adata.var["mt"].sum())
HAS_MITO = N_MITO > 0
print("Mitochondrial genes found:", N_MITO)

sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True)
assert {"n_genes_by_counts", "total_counts", "pct_counts_mt"} <= set(adata.obs.columns)
adata.obs[["n_genes_by_counts", "total_counts", "pct_counts_mt"]].describe()

Mitochondrial genes found: 0


,n_genes_by_counts,total_counts,pct_counts_mt
count,24673.000000,24673.000000,24673.0
mean,574.900985,1567.192627,0.0
std,206.621105,912.257385,0.0
min,12.000000,562.000000,0.0
25%,441.000000,975.000000,0.0
50%,519.000000,1246.000000,0.0
75%,652.000000,1816.000000,0.0
max,2757.000000,11816.000000,0.0


## 3. Per-sample QC limits

Each sample's limits come from its own cells: median ± 5 MADs of `log1p(genes per cell)` (plain MAD, no
normal-consistency scaling), turned back into gene counts with `ceil` (minimum) and `floor + 1` (maximum,
so a cell exactly at the bound is kept by `<`). If the MAD is 0 the sample gets no gene limits. The
% mitochondrial limit (median + 3 MADs, rounded up to 0.1) applies only when mitochondrial genes exist.

In [4]:
N_GENE_MADS, N_MT_MADS = 5, 3

def limits_for(cells):
    x = np.log1p(cells["n_genes_by_counts"].to_numpy(dtype=np.float64))
    m = float(np.median(x))
    mad = float(median_abs_deviation(x, scale=1.0))
    if mad == 0:
        min_genes, max_genes = 0, None
    else:
        min_genes = max(0, math.ceil(np.expm1(m - N_GENE_MADS * mad)))
        max_genes = math.floor(np.expm1(m + N_GENE_MADS * mad)) + 1
    row = {"log1p_genes_median": m, "log1p_genes_mad": mad, "min_genes": int(min_genes),
           "max_genes": None if max_genes is None else int(max_genes)}
    if HAS_MITO:
        pct = cells["pct_counts_mt"].to_numpy(dtype=np.float64)
        pm, pmad = float(np.median(pct)), float(median_abs_deviation(pct, scale=1.0))
        row.update(pct_mt_median=pm, pct_mt_mad=pmad, max_pct_mt=math.ceil((pm + N_MT_MADS * pmad) * 10) / 10)
    else:
        row.update(pct_mt_median=None, pct_mt_mad=None, max_pct_mt=None)
    return row

LIMITS = {s: limits_for(adata.obs[adata.obs["sample"] == s]) for s in adata.obs["sample"].cat.categories}
assert len(LIMITS) == len(SAMPLES)
pd.DataFrame(LIMITS).T

,log1p_genes_median,log1p_genes_mad,min_genes,max_genes,pct_mt_median,pct_mt_mad,max_pct_mt
patient_1016 · ctrl,6.152733,0.156281,215.0,1026.0,NaN,NaN,NaN
patient_1256 · ctrl,6.249975,0.179491,211.0,1270.0,NaN,NaN,NaN
patient_1488 · ctrl,6.171701,0.154449,221.0,1036.0,NaN,NaN,NaN
patient_1039 · ctrl,6.298949,0.215763,184.0,1600.0,NaN,NaN,NaN
patient_1015 · ctrl,6.270988,0.191055,203.0,1375.0,NaN,NaN,NaN
patient_107 · ctrl,6.297109,0.208064,191.0,1536.0,NaN,NaN,NaN
patient_1244 · ctrl,6.240276,0.162634,227.0,1156.0,NaN,NaN,NaN
patient_101 · ctrl,6.282267,0.238448,162.0,1762.0,NaN,NaN,NaN
patient_101 · stim,6.304449,0.220998,181.0,1651.0,NaN,NaN,NaN
patient_1488 · stim,6.240276,0.159982,230.0,1141.0,NaN,NaN,NaN


## 4. Filtering

In the spec's order: (1) cells below their own sample's minimum genes, (2) genes detected in fewer than
3 of the remaining cells, (3) cells at or above their own sample's maximum genes (and % mitochondrial
limit, if any). All cell checks use the section 2 metrics.

In [5]:
sample_of_cell = adata.obs["sample"].astype(str)
min_per_cell = sample_of_cell.map({s: l["min_genes"] for s, l in LIMITS.items()}).to_numpy(dtype=np.int64)
pass_min = adata.obs["n_genes_by_counts"].to_numpy() >= min_per_cell
step1 = adata[pass_min].copy()

detected_in = np.asarray((step1.X > 0).sum(axis=0)).ravel()
keep_gene = detected_in >= 3
step2 = step1[:, keep_gene].copy()

# Cross-check with Scanpy's own gene filter (same rule).
_check = step1.copy()
sc.pp.filter_genes(_check, min_cells=3)
assert list(_check.var_names) == list(step2.var_names)
del _check

s2_sample = step2.obs["sample"].astype(str)
max_per_cell = s2_sample.map({s: (np.inf if l["max_genes"] is None else l["max_genes"])
                              for s, l in LIMITS.items()}).to_numpy(dtype=np.float64)
pass_max = step2.obs["n_genes_by_counts"].to_numpy() < max_per_cell
if HAS_MITO:
    mt_per_cell = s2_sample.map({s: l["max_pct_mt"] for s, l in LIMITS.items()}).to_numpy(dtype=np.float64)
    pass_max &= step2.obs["pct_counts_mt"].to_numpy() < mt_per_cell
filtered = step2[pass_max].copy()

KEPT_PER_SAMPLE = []
for s in SAMPLES:
    name = s["name"]
    KEPT_PER_SAMPLE.append({
        "sample": name,
        "cells_before": s["cells_loaded"],
        "removed_min_genes": int(((adata.obs["sample"] == name) & ~pass_min).sum()),
        "removed_max_limits": int(((step2.obs["sample"] == name).to_numpy() & ~pass_max).sum()),
        "cells_after": int((filtered.obs["sample"] == name).sum()),
    })
for r in KEPT_PER_SAMPLE:
    assert r["cells_before"] - r["removed_min_genes"] - r["removed_max_limits"] == r["cells_after"]
assert sum(r["cells_after"] for r in KEPT_PER_SAMPLE) == filtered.n_obs
N_CELLS_KEPT, N_GENES_KEPT = filtered.shape
print(f"Kept {N_CELLS_KEPT:,} of {N_CELLS_LOADED:,} cells and {N_GENES_KEPT:,} of {N_GENES_LOADED:,} genes")
pd.DataFrame(KEPT_PER_SAMPLE)

Kept 24,296 of 24,673 cells and 15,700 of 15,706 genes


,sample,cells_before,removed_min_genes,removed_max_limits,cells_after
0,patient_1016 · ctrl,1723,1,46,1676
1,patient_1256 · ctrl,2108,0,19,2089
2,patient_1488 · ctrl,2031,1,50,1980
3,patient_1039 · ctrl,461,53,0,408
4,patient_1015 · ctrl,2738,0,17,2721
5,patient_107 · ctrl,517,0,0,517
6,patient_1244 · ctrl,1879,0,28,1851
7,patient_101 · ctrl,858,0,1,857
8,patient_101 · stim,1166,0,3,1163
9,patient_1488 · stim,2549,2,46,2501


## 5. Normalization and highly variable genes

Raw counts are kept in `layers["counts"]`. `normalize_total(target_sum=None)` scales each cell to the
**median total counts** of the kept cells; then `log1p`. Highly variable genes (Seurat flavor, cut-offs
0.0125 / 3 / 0.5) are chosen two ways: **pooled** (all cells together) and **per sample** (`batch_key="sample"`:
Scanpy computes dispersions within each sample and combines them, so genes that vary only *between*
samples are less likely to be picked). Each selection runs on its own copy so the log-normalized data
stays free of HVG columns.

In [6]:
lognorm = filtered.copy()
lognorm.layers["counts"] = lognorm.X.copy()
cell_totals = np.asarray(lognorm.X.sum(axis=1)).ravel()
assert np.all(cell_totals > 0)
NORMALIZE_TARGET = float(np.median(cell_totals))
sc.pp.normalize_total(lognorm, target_sum=None)
assert np.allclose(np.asarray(lognorm.X.sum(axis=1)).ravel(), NORMALIZE_TARGET, rtol=1e-4)
sc.pp.log1p(lognorm)
print("Normalized to the median total counts:", NORMALIZE_TARGET)

HVG_ARGS = dict(flavor="seurat", min_mean=0.0125, max_mean=3, min_disp=0.5)

def hvg_list(batch_key):
    tmp = lognorm.copy()
    sc.pp.highly_variable_genes(tmp, batch_key=batch_key, **HVG_ARGS)
    return list(tmp.var_names[tmp.var["highly_variable"].to_numpy(dtype=bool)])

HVG = {"pooled": hvg_list(None), "per_sample": hvg_list("sample")}
HVG_OVERLAP = len(set(HVG["pooled"]) & set(HVG["per_sample"]))
assert "highly_variable" not in lognorm.var.columns
assert all(len(v) > 0 for v in HVG.values())
print({k: len(v) for k, v in HVG.items()}, "shared:", HVG_OVERLAP)

Normalized to the median total counts: 1237.0


{'pooled': 1292, 'per_sample': 1094} shared: 933


## 7 (defined here). The sample mixing score

For each cell: its 30 nearest neighbors (exact Euclidean search, the cell itself excluded), the entropy
of their sample labels, divided by the largest entropy possible here, `min(entropy of the samples' overall
shares, ln 30)`, clipped to [0, 1]. 1 = the neighbors come from the samples in about their overall
proportions; 0 = all neighbors come from the cell's own sample.

In [7]:
MIX_K = 30
MIX_WARNINGS = []

def mixing_score(coords, batches, k=MIX_K):
    coords = np.asarray(coords)
    n = coords.shape[0]
    codes = pd.Categorical(batches).remove_unused_categories()
    b = np.asarray(codes.codes)
    nb = len(codes.categories)
    _, idx = NearestNeighbors(n_neighbors=k + 1, algorithm="brute", metric="euclidean").fit(coords).kneighbors(coords)
    self_mask = idx == np.arange(n)[:, None]
    has_self = self_mask.any(axis=1)
    if not has_self.all():  # only possible with many identical points; then drop the farthest one
        MIX_WARNINGS.append(f"{int((~has_self).sum())} cells did not find themselves among {k + 1} neighbors")
        self_mask[~has_self, -1] = True
    assert (self_mask.sum(axis=1) == 1).all()
    nbrs = idx[~self_mask].reshape(n, k)
    counts = np.zeros((n, nb))
    np.add.at(counts, (np.repeat(np.arange(n), k), b[nbrs].ravel()), 1)
    p = counts / k
    with np.errstate(divide="ignore", invalid="ignore"):
        H = -np.where(p > 0, p * np.log(p), 0.0).sum(axis=1)
    q = np.bincount(b, minlength=nb) / n
    q = q[q > 0]
    h_max = min(float(-(q * np.log(q)).sum()), math.log(k))
    assert h_max > 0, "mixing needs at least two samples"
    return np.clip(H / h_max, 0.0, 1.0)

# Tiny known-answer checks of the score.
_two = np.array([[0.0]] * 40 + [[100.0]] * 40)
assert np.allclose(mixing_score(_two, ["a"] * 40 + ["b"] * 40, k=10), 0.0)    # fully separated
_alt = np.arange(80, dtype=float)[:, None]
_s = mixing_score(_alt, ["a", "b"] * 40, k=10)
assert np.all((_s > 0.9) & (_s <= 1.0))                                          # interleaved
MIX_WARNINGS.clear()  # the identical points above trigger the tie warning on purpose; not about the data

## 6. Four runs: {pooled, per-sample HVGs} × {no correction, Harmony}

Each run starts from the log-normalized data: `raw` = all genes, keep that variant's highly variable genes,
`scale(max_value=10)` (no regression), PCA (50, arpack, seed 0), mixing before correction. In the Harmony
runs, **Harmony** (harmonypy, called directly) adjusts the 50 PCs so that cells from different samples that
look alike are pulled together; the corrected coordinates go to `obsm["X_pca_harmony"]` and get a second
mixing score. Then neighbors (15, on 50 dims), **Leiden before UMAP** (UMAP prunes the stored graph in place),
UMAP, Wilcoxon markers on `raw`, and composition (cells per sample and cluster).

In [8]:
RUN_RESULTS, RUN_OBS, RUN_OBSM, HARMONY_ORIENTATION = {}, {}, {}, {}

def run_variant(hvg_variant, harmony):
    name = f"{hvg_variant}_{'harmony' if harmony else 'none'}"
    a = lognorm.copy()
    a.raw = a
    a = a[:, a.var_names.isin(HVG[hvg_variant])].copy()
    assert a.n_vars == len(HVG[hvg_variant]) and a.raw.shape == lognorm.shape
    assert "highly_variable" not in a.var.columns   # PCA uses every gene in this object
    sc.pp.scale(a, max_value=10)
    sc.pp.pca(a, n_comps=50, svd_solver="arpack", random_state=SEED)
    assert a.obsm["X_pca"].shape == (a.n_obs, 50)

    mix_before = mixing_score(a.obsm["X_pca"][:, :50], a.obs["sample"])
    a.obs["mixing_before"] = mix_before
    rep = "X_pca"
    if harmony:
        ho = harmonypy.run_harmony(a.obsm["X_pca"][:, :50], a.obs[["sample"]], "sample",
                                   theta=2, max_iter_harmony=10, random_state=SEED)
        Z = np.asarray(ho.Z_corr)
        if Z.shape == (a.n_obs, 50):
            HARMONY_ORIENTATION[name] = "cells x PCs (used as is)"
        elif Z.shape == (50, a.n_obs):
            HARMONY_ORIENTATION[name] = "PCs x cells (transposed)"
            Z = Z.T
        else:
            raise ValueError(f"unexpected Z_corr shape {Z.shape}")
        a.obsm["X_pca_harmony"] = np.ascontiguousarray(Z)
        a.obs["mixing_after"] = mixing_score(a.obsm["X_pca_harmony"], a.obs["sample"])
        rep = "X_pca_harmony"

    sc.pp.neighbors(a, n_neighbors=15, n_pcs=50, use_rep=rep, random_state=SEED)
    sc.tl.leiden(a, resolution=1.0, flavor="leidenalg", n_iterations=-1, random_state=SEED,
                 directed=True, key_added="leiden")
    sc.tl.umap(a, random_state=SEED)
    sc.tl.rank_genes_groups(a, "leiden", method="wilcoxon", use_raw=True, pts=True, reference="rest",
                            corr_method="benjamini-hochberg", tie_correct=False)

    clusters = list(a.obs["leiden"].cat.categories)
    sizes = a.obs["leiden"].value_counts().reindex(clusters)
    assert int(sizes.sum()) == a.n_obs
    names = a.uns["rank_genes_groups"]["names"]
    top25 = {c: [str(g) for g in names[c][:25]] for c in clusters}

    ct = pd.crosstab(a.obs["sample"], a.obs["leiden"]).reindex(
        index=a.obs["sample"].cat.categories, columns=clusters, fill_value=0)
    share = ct.div(ct.sum(axis=1), axis=0)
    composition = [{"sample": s, "cluster": c, "n_cells": int(ct.loc[s, c]), "share": float(share.loc[s, c])}
                   for s in ct.index for c in clusters]

    per_cluster = a.obs.groupby("leiden", observed=False)
    mix_cluster = {c: {"n_cells": int(sizes[c]),
                       "mixing_before": float(per_cluster["mixing_before"].mean()[c])} for c in clusters}
    if harmony:
        for c in clusters:
            mix_cluster[c]["mixing_after"] = float(per_cluster["mixing_after"].mean()[c])

    result = {
        "hvg_variant": hvg_variant, "batch_correction": "harmony" if harmony else "none",
        "n_hvg": int(a.n_vars),
        "pca_variance_ratio": [float(v) for v in a.uns["pca"]["variance_ratio"]],
        "mixing_before_mean": float(a.obs["mixing_before"].mean()),
        "n_clusters": len(clusters),
        "cluster_sizes": {c: int(sizes[c]) for c in clusters},
        "top25_markers": top25,
        "composition": composition,
        "mixing_per_cluster": mix_cluster,
        "ari_vs_authors_cell_type": float(adjusted_rand_score(a.obs["cell_type"].astype(str),
                                                                 a.obs["leiden"].astype(str))),
    }
    if harmony:
        result["mixing_after_mean"] = float(a.obs["mixing_after"].mean())
        result["harmony_z_corr_orientation"] = HARMONY_ORIENTATION[name]

    RUN_RESULTS[name] = result
    RUN_OBS[f"leiden_{name}"] = a.obs["leiden"].copy()
    RUN_OBS[f"mixing_before_{name}"] = a.obs["mixing_before"].to_numpy()
    if harmony:
        RUN_OBS[f"mixing_after_{name}"] = a.obs["mixing_after"].to_numpy()
        RUN_OBSM[f"X_pca_harmony_{name}"] = a.obsm["X_pca_harmony"]
    RUN_OBSM[f"X_pca_{name}"] = a.obsm["X_pca"]
    RUN_OBSM[f"X_umap_{name}"] = a.obsm["X_umap"]
    assert list(a.obs_names) == list(lognorm.obs_names)
    msg = (f"{name}: {result['n_hvg']} genes, {result['n_clusters']} clusters, "
           f"mixing before {result['mixing_before_mean']:.4f}")
    if harmony:
        msg += f", after {result['mixing_after_mean']:.4f}"
    print(msg + f", ARI {result['ari_vs_authors_cell_type']:.4f}")
    return a

In [9]:
_ = run_variant("pooled", harmony=False)

/opt/anaconda3/envs/scrna/lib/python3.11/functools.py:909: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


/opt/anaconda3/envs/scrna/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/var/folders/07/3l4k_4wn505bhm3d0hvndqqc0000gn/T/ipykernel_37291/720735739.py:33: FutureWarning: In the future, the default backend for leiden will be igraph instead of leidenalg.

 To achieve the future defaults please pass: flavor="igraph" and n_iterations=2.  directed must also be False to work with igraph's implementation.
  sc.tl.leiden(a, resolution=1.0, flavor="leidenalg", n_iterations=-1, random_state=SEED,


pooled_none: 1292 genes, 18 clusters, mixing before 0.6527, ARI 0.3619


In [10]:
_ = run_variant("pooled", harmony=True)

/opt/anaconda3/envs/scrna/lib/python3.11/functools.py:909: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


2026-11-11 22:51:51,144 - harmonypy - INFO - Running Harmony


2026-11-11 22:51:51,144 - harmonypy - INFO -   Parameters:


2026-11-11 22:51:51,144 - harmonypy - INFO -     max_iter_harmony: 10


2026-11-11 22:51:51,144 - harmonypy - INFO -     max_iter_kmeans: 4


2026-11-11 22:51:51,144 - harmonypy - INFO -     epsilon_cluster: 0.001


... (18 outputs left out)

2026-11-11 22:51:51,304 - harmonypy - INFO - Iteration 7 of 10


2026-11-11 22:51:51,324 - harmonypy - INFO - Iteration 8 of 10


2026-11-11 22:51:51,344 - harmonypy - INFO - Converged after 8 iterations


pooled_harmony: 1292 genes, 13 clusters, mixing before 0.6527, after 0.8207, ARI 0.4582


In [11]:
_ = run_variant("per_sample", harmony=False)

/opt/anaconda3/envs/scrna/lib/python3.11/functools.py:909: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


per_sample_none: 1094 genes, 16 clusters, mixing before 0.6683, ARI 0.4040


In [12]:
_ = run_variant("per_sample", harmony=True)

/opt/anaconda3/envs/scrna/lib/python3.11/functools.py:909: UserWarning: zero-centering a sparse array/matrix densifies it.
  return dispatch(args[0].__class__)(*args, **kw)


2026-11-11 22:52:36,091 - harmonypy - INFO - Running Harmony


2026-11-11 22:52:36,091 - harmonypy - INFO -   Parameters:


2026-11-11 22:52:36,091 - harmonypy - INFO -     max_iter_harmony: 10


2026-11-11 22:52:36,091 - harmonypy - INFO -     max_iter_kmeans: 4


2026-11-11 22:52:36,092 - harmonypy - INFO -     epsilon_cluster: 0.001


... (17 outputs left out)

2026-11-11 22:52:36,230 - harmonypy - INFO - Iteration 6 of 10


2026-11-11 22:52:36,251 - harmonypy - INFO - Iteration 7 of 10


2026-11-11 22:52:36,271 - harmonypy - INFO - Converged after 7 iterations


per_sample_harmony: 1094 genes, 14 clusters, mixing before 0.6683, after 0.8199, ARI 0.5313


## 8. Biological checks (reported, never used to choose anything)

1. Adjusted Rand index between each run's clusters and the authors' cell types (reported above, collected below).
2. Interferon response: for each authors' cell type with at least 50 kept cells in **both** conditions, the mean
   log-normalized expression (all genes) of ISG15, IFI6, IFIT1, IFIT3 and MX1 in stim minus ctrl. Expected
   direction: higher in stim. Also which of these genes are in each highly variable gene list.

In [13]:
ISG = [g for g in SETTINGS["checks"]["isg_genes"] if g in lognorm.var_names]
MIN_CELLS = SETTINGS["checks"]["min_cells_per_condition"]
cond = lognorm.obs["condition"].astype(str).to_numpy()
ctype = lognorm.obs["cell_type"].astype(str).to_numpy()
expr = lognorm[:, ISG].X
expr = (expr.toarray() if sp.issparse(expr) else np.asarray(expr)).astype(np.float64)

IFN_TABLE = []
for t in lognorm.obs["cell_type"].cat.categories:
    in_t = ctype == t
    n_ctrl, n_stim = int((in_t & (cond == "ctrl")).sum()), int((in_t & (cond == "stim")).sum())
    if n_ctrl < MIN_CELLS or n_stim < MIN_CELLS:
        continue
    for j, g in enumerate(ISG):
        m_ctrl = float(expr[in_t & (cond == "ctrl"), j].mean())
        m_stim = float(expr[in_t & (cond == "stim"), j].mean())
        IFN_TABLE.append({"cell_type": str(t), "gene": g, "n_ctrl": n_ctrl, "n_stim": n_stim,
                          "mean_ctrl": m_ctrl, "mean_stim": m_stim, "stim_minus_ctrl": m_stim - m_ctrl})
ISG_IN_HVG = {k: [g for g in ISG if g in set(v)] for k, v in HVG.items()}
print("Interferon genes present:", ISG, "| in HVG lists:", ISG_IN_HVG)
ifn_df = pd.DataFrame(IFN_TABLE)
print("Rows with stim > ctrl:", int((ifn_df["stim_minus_ctrl"] > 0).sum()), "of", len(ifn_df))
ifn_df.pivot(index="cell_type", columns="gene", values="stim_minus_ctrl").round(3)

Interferon genes present: ['ISG15', 'IFI6', 'IFIT1', 'IFIT3', 'MX1'] | in HVG lists: {'pooled': ['ISG15', 'IFIT1', 'IFIT3'], 'per_sample': ['ISG15', 'IFIT3']}
Rows with stim > ctrl: 35 of 35


gene,IFI6,IFIT1,IFIT3,ISG15,MX1
cell_type,,,,,
B cells,1.310,1.049,1.375,2.160,1.020
CD14+ Monocytes,1.320,1.511,1.614,3.768,0.998
CD4 T cells,1.221,0.882,1.009,1.796,0.803
CD8 T cells,1.409,0.994,1.093,1.883,0.892
Dendritic cells,1.314,1.188,1.428,3.155,1.311
FCGR3A+ Monocytes,1.199,1.817,1.597,3.271,1.025
NK cells,1.564,1.085,1.374,2.190,0.973


In [14]:
pd.DataFrame({k: {"clusters": v["n_clusters"], "mixing before": v["mixing_before_mean"],
                  "mixing after": v.get("mixing_after_mean"), "ARI": v["ari_vs_authors_cell_type"]}
              for k, v in RUN_RESULTS.items()}).T

,clusters,mixing before,mixing after,ARI
pooled_none,18.0,0.652744,NaN,0.361856
pooled_harmony,13.0,0.652744,0.820738,0.458223
per_sample_none,16.0,0.668326,NaN,0.403982
per_sample_harmony,14.0,0.668326,0.819937,0.531256


## 9. Save

`kang_summary.json` (plain JSON, full precision) and `kang_results.h5ad` (no expression matrix; per-run clusters,
mixing scores, PCA / Harmony / UMAP coordinates, and the summary as JSON text in `uns["kang_validation"]`).

In [15]:
assert set(RUN_RESULTS) == {"pooled_none", "pooled_harmony", "per_sample_none", "per_sample_harmony"}
summary = {
    "source_notebook": "reference/kang-multisample-validated.ipynb",
    "spec": "docs/kang_validation_spec.md",
    "date": datetime.date.today().isoformat(),
    "input": {"file": DATA_FILE.name, "sha256": INPUT_SHA256, "n_cells": int(N_CELLS_LOADED),
              "n_genes": int(N_GENES_LOADED), "gene_names_were_unique": names_were_unique},
    "seed": SEED,
    "settings": SETTINGS,
    "choices": CHOICES,
    "versions": VERSIONS,
    "samples": SAMPLES,
    "n_mito_genes": N_MITO,
    "qc_limits": [{"sample": s, **l} for s, l in LIMITS.items()],
    "cells_kept_per_sample": KEPT_PER_SAMPLE,
    "n_cells_kept": int(N_CELLS_KEPT),
    "n_genes_kept": int(N_GENES_KEPT),
    "normalize_target_sum_used": NORMALIZE_TARGET,
    "hvg": {"pooled": sorted(HVG["pooled"]), "per_sample": sorted(HVG["per_sample"]),
            "n_pooled": len(HVG["pooled"]), "n_per_sample": len(HVG["per_sample"]), "n_overlap": HVG_OVERLAP},
    "runs": RUN_RESULTS,
    "interferon": {"genes": ISG, "in_hvg": ISG_IN_HVG, "table": IFN_TABLE},
    "warnings": MIX_WARNINGS,
}
summary_text = json.dumps(summary, indent=2, ensure_ascii=False, allow_nan=False)
(OUT_DIR / "kang_summary.json").write_text(summary_text + "\n", encoding="utf-8")

obs = lognorm.obs[["sample", "condition", "cell_type"]].copy()
for key, values in RUN_OBS.items():
    obs[key] = values.to_numpy() if hasattr(values, "to_numpy") else values
    if key.startswith("leiden_"):
        obs[key] = pd.Categorical(obs[key], categories=list(values.cat.categories))
results = ad.AnnData(obs=obs, obsm={k: np.asarray(v) for k, v in RUN_OBSM.items()})
results.uns["kang_validation"] = summary_text
results.write_h5ad(OUT_DIR / "kang_results.h5ad", compression="gzip")

back = ad.read_h5ad(OUT_DIR / "kang_results.h5ad")
assert back.n_obs == N_CELLS_KEPT and json.loads(back.uns["kang_validation"]) == summary
print("Saved", sorted(p.name for p in OUT_DIR.iterdir()))
print("summary SHA-256:", hashlib.sha256((OUT_DIR / "kang_summary.json").read_bytes()).hexdigest())

Saved ['kang_results.h5ad', 'kang_summary.json']
summary SHA-256: 6c5c14f693fc1246f23bded0a835d4630473d6b0e10b6aaaa8344824b88d967d
